In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy rustworkx scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Kvantový aproximační optimalizační algoritmus

*Odhadovaná doba použití: 22 minut na procesoru Heron r3 (POZNÁMKA: Jedná se pouze o odhad. Skutečná doba běhu se může lišit.)*
## Pozadí
Tento tutoriál ukazuje, jak implementovat **Kvantový aproximační optimalizační algoritmus (QAOA)** – hybridní (kvantově-klasickou) iterativní metodu – v kontextu Qiskit vzorů. Nejprve vyřešíš problém **maximálního řezu** (neboli **Max-Cut**) pro malý graf a poté se naučíš, jak ho spustit v užitkové škále. Všechna hardwarová spuštění v tutoriálu by měla proběhnout v časovém limitu volně dostupného Open Plánu.

Problém Max-Cut je optimalizační problém, který je obtížné vyřešit (konkrétněji jde o *NP-těžký* problém) s řadou různých aplikací v oblasti clusterování, síťové vědy a statistické fyziky. Tento tutoriál uvažuje graf uzlů propojených hranami a snaží se rozdělit uzly do dvou množin tak, aby byl maximalizován počet hran, které tento řez překračuje.

![Ilustrace problému max-cut](../docs/images/tutorials/quantum-approximate-optimization-algorithm/maxcut-illustration.avif)
## Požadavky
Před zahájením tohoto tutoriálu se ujisti, že máš nainstalováno následující:
- Qiskit SDK v1.0 nebo novější s podporou [vizualizace](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 nebo novější (`pip install qiskit-ibm-runtime`)

Kromě toho budeš potřebovat přístup k instanci na [IBM Quantum Platform](/guides/cloud-setup). Vezmi na vědomí, že tento tutoriál nelze spustit na Open Plánu, protože používá workloady prostřednictvím [sessions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session), které jsou dostupné pouze s přístupem Premium Plánu.
## Nastavení

In [1]:
import matplotlib
import matplotlib.pyplot as plt
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
from scipy.optimize import minimize
from collections import defaultdict
from typing import Sequence


from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Část I. QAOA v malém měřítku
První část tohoto tutoriálu používá problém Max-Cut v malém měřítku k ilustraci kroků pro řešení optimalizačního problému pomocí kvantového počítače.

Aby sis lépe porozuměl kontextu před mapováním tohoto problému na kvantový algoritmus, lze lépe pochopit, jak se problém Max-Cut stává klasickým kombinatorickým optimalizačním problémem, když nejprve uvažujeme minimalizaci funkce $f(x)$

$$
\min_{x\in {0, 1}^n}f(x),
$$

kde vstup $x$ je vektor, jehož složky odpovídají jednotlivým uzlům grafu. Pak omez každou z těchto složek na hodnotu $0$ nebo $1$ (které představují zahrnutí nebo nezahrnutí do řezu). Tento příklad v malém měřítku používá graf s $n=5$ uzly.

Mohl bys napsat funkci dvojice uzlů $i,j$, která indikuje, zda odpovídající hrana $(i,j)$ leží v řezu. Například funkce $x_i + x_j - 2 x_i x_j$ je 1, pouze pokud jedno z $x_i$ nebo $x_j$ je 1 (což znamená, že hrana leží v řezu), a jinak je nulová. Problém maximalizace hran v řezu lze formulovat jako

$$
\max_{x\in {0, 1}^n} \sum_{(i,j)} x_i + x_j - 2 x_i x_j,
$$

což lze přepsat jako minimalizaci ve tvaru

$$
\min_{x\in {0, 1}^n} \sum_{(i,j)}  2 x_i x_j - x_i - x_j.
$$

Minimum $f(x)$ v tomto případě nastane, když je počet hran překračovaných řezem maximální. Jak vidíš, dosud tu není nic, co by se týkalo kvantového počítání. Je potřeba přeformulovat tento problém do podoby, které kvantový počítač rozumí.
Inicializuj svůj problém vytvořením grafu s $n=5$ uzly.

In [2]:
n = 5

graph = rx.PyGraph()
graph.add_nodes_from(np.arange(0, n, 1))
edge_list = [
    (0, 1, 1.0),
    (0, 2, 1.0),
    (0, 4, 1.0),
    (1, 2, 1.0),
    (2, 3, 1.0),
    (3, 4, 1.0),
]
graph.add_edges_from(edge_list)
draw_graph(graph, node_size=600, with_labels=True)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif)

### Krok 1: Mapování klasických vstupů na kvantový problém
Prvním krokem vzoru je mapování klasického problému (grafu) na kvantové **Circuit** a **operátory**. K tomu jsou potřeba tři hlavní kroky:

1. Využít řadu matematických přeformulování k reprezentaci tohoto problému pomocí notace problémů Kvadratické neomezené binární optimalizace (QUBO).
2. Přepsat optimalizační problém jako Hamiltonián, jehož základní stav odpovídá řešení minimalizujícímu nákladovou funkci.
3. Vytvořit kvantový Circuit, který připraví základní stav tohoto Hamiltoniánu procesem podobným kvantovému žíhání.

**Poznámka:** V metodologii QAOA chceš nakonec mít operátor (**Hamiltonián**), který reprezentuje **nákladovou funkci** našeho hybridního algoritmu, a také parametrizovaný Circuit (**Ansatz**), který reprezentuje kvantové stavy s kandidátními řešeními problému. Z těchto kandidátních stavů lze vzorkovat a poté je vyhodnocovat pomocí nákladové funkce.

#### Graf &rarr; optimalizační problém
Prvním krokem mapování je změna notace. Následující výraz vyjadřuje problém v QUBO notaci:

$$
\min_{x\in {0, 1}^n}x^T Q x,
$$

kde $Q$ je matice $n\times n$ reálných čísel, $n$ odpovídá počtu uzlů v grafu, $x$ je vektor binárních proměnných zavedených výše a $x^T$ označuje transpozici vektoru $x$.

```
Maximize
 -2*x_0*x_1 - 2*x_0*x_2 - 2*x_0*x_4 - 2*x_1*x_2 - 2*x_2*x_3 - 2*x_3*x_4 + 3*x_0
 + 2*x_1 + 3*x_2 + 2*x_3 + 2*x_4

Subject to
  No constraints

  Binary variables (5)
    x_0 x_1 x_2 x_3 x_4
```
### Optimalizační problém &rarr; Hamiltonián
Poté lze QUBO problém přeformulovat jako **Hamiltonián** (zde matice reprezentující energii systému):

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

<details>
<summary>
**Kroky přeformulování z QAOA problému na Hamiltonián**
</summary>

Abychom ukázali, jak lze QAOA problém přepsat tímto způsobem, nejprve nahraď binární proměnné $x_i$ novou sadou proměnných $z_i\in{-1, 1}$ pomocí

$$
x_i = \frac{1-z_i}{2}.
$$

Zde vidíš, že pokud je $x_i$ rovno $0$, pak $z_i$ musí být $1$. Když se $x_i$ nahradí $z_i$ v optimalizačním problému ($x^TQx$), lze získat ekvivalentní formulaci.

$$
x^TQx=\sum_{ij}Q_{ij}x_ix_j \\ =\frac{1}{4}\sum_{ij}Q_{ij}(1-z_i)(1-z_j) \\=\frac{1}{4}\sum_{ij}Q_{ij}z_iz_j-\frac{1}{4}\sum_{ij}(Q_{ij}+Q_{ji})z_i + \frac{n^2}{4}.
$$

Pokud nyní definujeme $b_i=-\sum_{j}(Q_{ij}+Q_{ji})$, odstraníme prefaktor a konstantní člen $n^2$, dostaneme dvě ekvivalentní formulace stejného optimalizačního problému.

$$
\min_{x\in{0,1}^n} x^TQx\Longleftrightarrow \min_{z\in{-1,1}^n}z^TQz + b^Tz
$$

Zde $b$ závisí na $Q$. Všimni si, že pro získání $z^TQz + b^Tz$ jsme vypustili faktor 1/4 a konstantní posun $n^2$, které nehrají roli v optimalizaci.

Nyní, pro získání kvantové formulace problému, povyš proměnné $z_i$ na Pauliho matici $Z$, jako je matice $2\times 2$ ve tvaru

$$
Z_i = \begin{pmatrix}1 & 0 \\ 0 & -1\end{pmatrix}.
$$

Když tyto matice dosadíš do výše uvedeného optimalizačního problému, získáš následující Hamiltonián

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

*Také si vzpomeň, že matice $Z$ jsou vloženy do výpočetního prostoru kvantového počítače, tedy do Hilbertova prostoru velikosti $2^n\times 2^n$. Proto bys měl chápat výrazy jako $Z_iZ_j$ jako tenzorový součin $Z_i\otimes Z_j$ vložený do Hilbertova prostoru $2^n\times 2^n$. Například v problému s pěti rozhodovacími proměnnými je výraz $Z_1Z_3$ chápán jako $I\otimes Z_3\otimes I\otimes Z_1\otimes I$, kde $I$ je jednotková matice $2\times 2$.*
</details>

Tento Hamiltonián se nazývá **Hamiltonián nákladové funkce**. Má tu vlastnost, že jeho základní stav odpovídá řešení, které **minimalizuje nákladovou funkci $f(x)$**.
Aby sis tedy vyřešil optimalizační problém, potřebuješ nyní připravit základní stav $H_C$ (nebo stav s vysokým překryvem s ním) na kvantovém počítači. Vzorkování z tohoto stavu pak s vysokou pravděpodobností přinese řešení $min~f(x)$.
Nyní uvažme Hamiltonián $H_C$ pro problém **Max-Cut**. Přiřaď každý vrchol grafu ke Qubitu ve stavu $|0\rangle$ nebo $|1\rangle$, kde hodnota označuje množinu, do které vrchol patří. Cílem problému je maximalizovat počet hran $(v_1, v_2)$, pro které platí $v_1 = |0\rangle$ a $v_2 = |1\rangle$, nebo naopak. Pokud přiřadíme operátor $Z$ každému Qubitu, kde

$$
    Z|0\rangle = |0\rangle \qquad Z|1\rangle = -|1\rangle
$$

pak hrana $(v_1, v_2)$ patří do řezu, pokud vlastní hodnota $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = -1$; jinými slovy, Qubity přiřazené k $v_1$ a $v_2$ jsou různé. Podobně $(v_1, v_2)$ nepatří do řezu, pokud vlastní hodnota $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = 1$. Všimni si, že nás nezajímá přesný stav Qubitu přiřazený každému vrcholu, ale pouze to, zda jsou na hraně stejné nebo různé. Problém Max-Cut vyžaduje najít přiřazení Qubitů na vrcholech tak, aby byla minimalizována vlastní hodnota následujícího Hamiltoniánu
$$
    H_C = \sum_{(i,j) \in e} Q_{ij} \cdot Z_i Z_j.
$$

Jinými slovy, $b_i = 0$ pro všechna $i$ v problému Max-Cut. Hodnota $Q_{ij}$ označuje váhu hrany. V tomto tutoriálu uvažujeme neváhovaný graf, tedy $Q_{ij} = 1.0$ pro všechna $i, j$.

In [ ]:
def build_max_cut_paulis(
    graph: rx.PyGraph,
) -> list[tuple[str, list[int], float]]:
    """Convert the graph to Pauli list.

    This function does the inverse of `build_max_cut_graph`
    """
    pauli_list = []
    for edge in list(graph.edge_list()):
        weight = graph.get_edge_data(edge[0], edge[1])
        pauli_list.append(("ZZ", [edge[0], edge[1]], weight))
    return pauli_list


max_cut_paulis = build_max_cut_paulis(graph)
cost_hamiltonian = SparsePauliOp.from_sparse_list(max_cut_paulis, n)
print("Cost Function Hamiltonian:", cost_hamiltonian)

Cost Function Hamiltonian: SparsePauliOp(['IIIZZ', 'IIZIZ', 'ZIIIZ', 'IIZZI', 'IZZII', 'ZZIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])


#### Hamiltonian &rarr; quantum circuit

The Hamiltonian $H_C$ contains the quantum definition of your problem. Now you can create a quantum circuit that will help *sample* good solutions from the quantum computer. The QAOA is inspired by quantum annealing and applies alternating layers of operators in the quantum circuit.

The general idea is to start in the ground state of a known system, $H^{\otimes n}|0\rangle$ above, and then steer the system into the ground state of the cost operator that you are interested in. This is done by applying the operators $\exp\{-i\gamma_k H_C\}$ and $\exp\{-i\beta_k H_m\}$ with angles $\gamma_1,...,\gamma_p$ and $\beta_1,...,\beta_p~$.


The quantum circuit that you generate is **parametrized** by $\gamma_i$ and $\beta_i$, so you can try out different values of $\gamma_i$ and $\beta_i$ and sample from the resulting state.

![Circuit diagram with QAOA layers](../docs/images/tutorials/quantum-approximate-optimization-algorithm/circuit-diagram.svg)


In this case, you will try an example with one QAOA layer that contains two parameters: $\gamma_1$ and $\beta_1$.

In [4]:
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=2)
circuit.measure_all()

circuit.draw("mpl")

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/7bd8c6d4-f40f-4a11-a440-0b26d9021b53-0.avif" alt="Output of the previous code cell" />

In [5]:
circuit.parameters

ParameterView([ParameterVectorElement(β[0]), ParameterVectorElement(β[1]), ParameterVectorElement(γ[0]), ParameterVectorElement(γ[1])])

### Step 2: Optimize problem for quantum hardware execution

The circuit above contains a series of abstractions useful to think about quantum algorithms, but not possible to run on the hardware. To be able to run on a QPU, the circuit needs to undergo a series of operations that make up the **transpilation** or **circuit optimization** step of the pattern.

The Qiskit library offers a series of **transpilation passes** that cater to a wide range of circuit transformations. You need to make sure that your circuit is **optimized** for your purpose.

Transpilation may involves several steps, such as:

* **Initial mapping** of the qubits in the circuit (such as decision variables) to physical qubits on the device.
* **Unrolling** of the instructions in the quantum circuit to the hardware-native instructions that the backend understands.
* **Routing** of any qubits in the circuit that interact to physical qubits that are adjacent with one another.
* **Error suppression** by adding single-qubit gates to suppress noise with dynamical decoupling.


More information about transpilation is available in our [documentation](/docs/guides/transpile).

The following code transforms and optimizes the abstract circuit into a format that is ready for execution on one of devices accessible through the cloud using the **Qiskit IBM Runtime service**.

In [6]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
print(backend)

# Create pass manager for transpilation
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

candidate_circuit = pm.run(circuit)
candidate_circuit.draw("mpl", fold=False, idle_wires=False)

<IBMBackend('test_heron_pok_1')>


<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif" alt="Output of the previous code cell" />

### Step 3: Execute using Qiskit primitives

In the QAOA workflow, the optimal QAOA parameters are found in an iterative optimization loop, which runs a series of circuit evaluations and uses a classical optimizer to find the optimal $\beta_k$ and $\gamma_k$ parameters. This execution loop is executed via the following steps:

1. Define the initial parameters
2. Instantiate a new `Session` containing the optimization loop and the primitive used to sample the circuit
3. Once an optimal set of parameters is found, execute the circuit a final time to obtain a final distribution which will be used in the post-process step.

#### Define circuit with initial parameters
We start with arbitrary chosen parameters.

In [7]:
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_beta, initial_gamma, initial_gamma]

### Krok 2: Optimalizace problému pro spuštění na kvantovém hardwaru
Circuit výše obsahuje řadu abstrakcí užitečných pro přemýšlení o kvantových algoritmech, ale na hardwaru je nelze spustit. Aby bylo možné spustit na QPU, musí Circuit projít řadou operací, které tvoří krok **transpilace** nebo **optimalizace Circuit** vzoru.

Knihovna Qiskit nabízí řadu **transpilačních průchodů**, které pokrývají širokou škálu transformací Circuit. Je potřeba zajistit, aby byl tvůj Circuit **optimalizován** pro daný účel.

Transpilace může zahrnovat několik kroků, jako jsou:

* **Počáteční mapování** Qubitů v Circuit (jako jsou rozhodovací proměnné) na fyzické Qubity na zařízení.
* **Rozvinutí** instrukcí v kvantovém Circuit do nativních instrukcí hardwaru, kterým Backend rozumí.
* **Směrování** Qubitů v Circuit, které spolu interagují, na fyzické Qubity, které jsou vzájemně sousední.
* **Potlačení chyb** přidáním jednoQubitových Gate pro potlačení šumu pomocí dynamického oddělování.

Více informací o transpilaci najdeš v naší [dokumentaci](/guides/transpile).

Následující kód transformuje a optimalizuje abstraktní Circuit do formátu připraveného ke spuštění na jednom ze zařízení přístupných přes cloud pomocí **Qiskit IBM Runtime služby**.

In [8]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator):
    # transform the observable defined on virtual qubits to
    # an observable defined on all physical qubits
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub])

    results = job.result()[0]
    cost = results.data.evs

    objective_func_vals.append(cost)

    return cost

In [9]:
objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit, cost_hamiltonian, estimator),
        method="COBYLA",
        tol=1e-2,
    )
    print(result)

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -1.6295230263157894
       x: [ 1.530e+00  1.439e+00  4.071e+00  4.434e+00]
    nfev: 26
   maxcv: 0.0


![Výstup předchozí buňky kódu](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif)

### Krok 3: Spuštění pomocí Qiskit primitiv
V pracovním postupu QAOA jsou optimální parametry QAOA nalezeny v iterační optimalizační smyčce, která spouští řadu vyhodnocení Circuit a používá klasický optimalizátor k nalezení optimálních parametrů $\beta_k$ a $\gamma_k$. Tato spouštěcí smyčka se provádí pomocí následujících kroků:

1. Definuj počáteční parametry
2. Vytvoř novou `Session` obsahující optimalizační smyčku a primitivum použité pro vzorkování Circuit
3. Jakmile je nalezena optimální sada parametrů, spusť Circuit naposledy pro získání finálního rozdělení, které bude použito v kroku po-zpracování.
#### Definuj Circuit s počátečními parametry
Začínáme s libovolně zvolenými parametry.

In [10]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif" alt="Output of the previous code cell" />

#### Definuj Backend a spouštěcí primitivum
Použij **Qiskit Runtime primitiva** pro interakci s IBM&reg; Backendy. Dvě primitiva jsou Sampler a Estimator a volba primitiva závisí na tom, jaký typ měření chceš spustit na kvantovém počítači. Pro minimalizaci $H_C$ použij Estimator, protože měření nákladové funkce je jednoduše střední hodnota $\langle H_C \rangle$.
#### Spustit
Primitiva nabízejí různé [režimy spuštění](/guides/execution-modes) pro plánování workloadů na kvantových zařízeních a pracovní postup QAOA běží iterativně v Session.

![Ilustrace zobrazující chování režimů Single job, Batch a Session runtime.](../docs/images/tutorials/quantum-approximate-optimization-algorithm/runtime-modes.avif)

Funkci nákladů založenou na Sampleru můžeš zapojit do minimalizační rutiny SciPy pro nalezení optimálních parametrů.

In [11]:
optimized_circuit = candidate_circuit.assign_parameters(result.x)
optimized_circuit.draw("mpl", fold=False, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/2989e76e-4296-4dd8-b065-2b8fced064cf-0.avif" alt="Output of the previous code cell" />

In [12]:
# If using qiskit-ibm-runtime<0.24.0, change `mode=` to `backend=`
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"

pub = (optimized_circuit,)
job = sampler.run([pub], shots=int(1e4))
counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_int = {key: val / shots for key, val in counts_int.items()}
final_distribution_bin = {key: val / shots for key, val in counts_bin.items()}
print(final_distribution_int)

{28: 0.0328, 11: 0.0343, 2: 0.0296, 25: 0.0308, 16: 0.0303, 27: 0.0302, 13: 0.0323, 7: 0.0312, 4: 0.0296, 9: 0.0295, 26: 0.0321, 30: 0.031, 23: 0.0324, 31: 0.0303, 21: 0.0335, 15: 0.0317, 12: 0.0309, 29: 0.0297, 3: 0.0313, 5: 0.0312, 6: 0.0274, 10: 0.0329, 22: 0.0353, 0: 0.0315, 20: 0.0326, 8: 0.0322, 14: 0.0306, 17: 0.0295, 18: 0.0279, 1: 0.0325, 24: 0.0334, 19: 0.0295}


### Step 4: Post-process and return result in desired classical format

The post-processing step interprets the sampling output to return a solution for your original problem. In this case, you are interested in the bitstring with the highest probability as this determines the optimal cut. The symmetries in the problem allow for four possible solutions, and the sampling process will return one of them with a slightly higher probability, but you can see in the plotted distribution below that four of the bitstrings are distinctively more likely than the rest.

In [13]:
# auxiliary functions to sample most likely bitstring
def to_bitstring(integer, num_bits):
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]


keys = list(final_distribution_int.keys())
values = list(final_distribution_int.values())
most_likely = keys[np.argmax(np.abs(values))]
most_likely_bitstring = to_bitstring(most_likely, len(graph))
most_likely_bitstring.reverse()

print("Result bitstring:", most_likely_bitstring)

Result bitstring: [0, 1, 1, 0, 1]


In [14]:
matplotlib.rcParams.update({"font.size": 10})
final_bits = final_distribution_bin
values = np.abs(list(final_bits.values()))
top_4_values = sorted(values, reverse=True)[:4]
positions = []
for value in top_4_values:
    positions.append(np.where(values == value)[0])
fig = plt.figure(figsize=(11, 6))
ax = fig.add_subplot(1, 1, 1)
plt.xticks(rotation=45)
plt.title("Result Distribution")
plt.xlabel("Bitstrings (reversed)")
plt.ylabel("Probability")
ax.bar(list(final_bits.keys()), list(final_bits.values()), color="tab:grey")
for p in positions:
    ax.get_children()[int(p[0])].set_color("tab:purple")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif)

Jakmile jsi nalezl optimální parametry pro Circuit, můžeš je přiřadit a vzorkovat finální rozdělení získané s optimalizovanými parametry. Zde by mělo být použito primitivum *Sampler*, protože jde o rozdělení pravděpodobnosti bitřetězcových měření, která odpovídají optimálnímu řezu grafu.

**Poznámka:** To znamená připravit kvantový stav $\psi$ v počítači a poté ho změřit. Měření zkolabuje stav do jediného výpočetního základního stavu – například `010101110000...` – který odpovídá kandidátnímu řešení $x$ našeho původního optimalizačního problému ($\max f(x)$ nebo $\min f(x)$ v závislosti na úloze).

In [15]:
# auxiliary function to plot graphs
def plot_result(G, x):
    colors = ["tab:grey" if i == 0 else "tab:purple" for i in x]
    pos, _default_axes = rx.spring_layout(G), plt.axes(frameon=True)
    rx.visualization.mpl_draw(
        G, node_color=colors, node_size=100, alpha=0.8, pos=pos
    )


plot_result(graph, most_likely_bitstring)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif" alt="Output of the previous code cell" />

And calculate the value of the cut:

In [16]:
def evaluate_sample(x: Sequence[int], graph: rx.PyGraph) -> float:
    assert len(x) == len(
        list(graph.nodes())
    ), "The length of x must coincide with the number of nodes in the graph."
    return sum(
        x[u] * (1 - x[v]) + x[v] * (1 - x[u])
        for u, v in list(graph.edge_list())
    )


cut_value = evaluate_sample(most_likely_bitstring, graph)
print("The value of the cut is:", cut_value)

The value of the cut is: 5


## Part II. Scale it up!

You have access to many devices with over 100 qubits on IBM Quantum&reg; Platform. Select one on which to solve Max-Cut on a 100-node weighted graph. This is a "utility-scale" problem. The steps to build the workflow are followed the same as above, but with a much larger graph.

In [17]:
n = 100  # Number of nodes in graph
graph_100 = rx.PyGraph()
graph_100.add_nodes_from(np.arange(0, n, 1))
elist = []
for edge in backend.coupling_map:
    if edge[0] < n and edge[1] < n:
        elist.append((edge[0], edge[1], 1.0))
graph_100.add_edges_from(elist)
draw_graph(graph_100, node_size=200, with_labels=True, width=1)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/590fe2ce-0.avif" alt="Output of the previous code cell" />

### Krok 4: Po-zpracování a vrácení výsledku v požadovaném klasickém formátu
Krok po-zpracování interpretuje výstup vzorkování a vrací řešení původního problému. V tomto případě tě zajímá bitřetězec s nejvyšší pravděpodobností, protože ten určuje optimální řez. Symetrie v problému umožňují čtyři možná řešení a proces vzorkování vrátí jedno z nich s mírně vyšší pravděpodobností, ale v níže zobrazeném rozdělení vidíš, že čtyři z bitřetězců jsou zřetelně pravděpodobnější než ostatní.

In [18]:
max_cut_paulis_100 = build_max_cut_paulis(graph_100)

cost_hamiltonian_100 = SparsePauliOp.from_sparse_list(max_cut_paulis_100, 100)
print("Cost Function Hamiltonian:", cost_hamiltonian_100)

Cost Function Hamiltonian: SparsePauliOp(['...'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0

#### Hamiltonian &rarr; quantum circuit

In [19]:
circuit_100 = QAOAAnsatz(cost_operator=cost_hamiltonian_100, reps=1)
circuit_100.measure_all()

circuit_100.draw("mpl", fold=False, scale=0.2, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/9693adfc-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum execution
To scale the circuit optimization step to utility-scale problems, you can take advantage of the high performance transpilation strategies introduced in Qiskit SDK v1.0. Other tools include the new transpiler service with [AI enhanced transpiler passes](/docs/guides/ai-transpiler-passes).

In [20]:
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

candidate_circuit_100 = pm.run(circuit_100)
candidate_circuit_100.draw("mpl", fold=False, scale=0.1, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3a14e7ad-0.avif" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif)

#### Vizualizace nejlepšího řezu
Z optimálního bitřetězce pak můžeš vizualizovat tento řez na původním grafu.

In [21]:
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_gamma]

objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)

    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit_100, cost_hamiltonian_100, estimator),
        method="COBYLA",
    )
    print(result)

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -3.9939191365979383
       x: [ 1.571e+00  3.142e+00]
    nfev: 29
   maxcv: 0.0


![Výstup předchozí buňky kódu](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif)

A vypočítej hodnotu řezu:

In [22]:
optimized_circuit_100 = candidate_circuit_100.assign_parameters(result.x)
optimized_circuit_100.draw("mpl", fold=False, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/1c432c2e-0.avif" alt="Output of the previous code cell" />

Finally, execute the circuit with the optimal parameters to sample from the corresponding distribution.

In [23]:
# If using qiskit-ibm-runtime<0.24.0, change `mode=` to `backend=`
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"


pub = (optimized_circuit_100,)
job = sampler.run([pub], shots=int(1e4))

counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_100_int = {
    key: val / shots for key, val in counts_int.items()
}

## Část II. Škálování na větší problémy!
Máš přístup k mnoha zařízením s více než 100 qubity na platformě IBM Quantum&reg;. Vyber jedno, na kterém vyřešíš Max-Cut na váženém grafu se 100 uzly. Jedná se o problém v „utility-scale" měřítku. Kroky k sestavení pracovního postupu jsou stejné jako výše, ale s mnohem větším grafem.

In [24]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/0fda3611-0.avif" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/590fe2ce-0.avif)

### Krok 1: Mapování klasických vstupů na kvantový problém
#### Graf &rarr; Hamiltonián
Nejprve převeď graf, který chceš vyřešit, přímo na Hamiltonián vhodný pro QAOA.

In [25]:
_PARITY = np.array(
    [-1 if bin(i).count("1") % 2 else 1 for i in range(256)],
    dtype=np.complex128,
)


def evaluate_sparse_pauli(state: int, observable: SparsePauliOp) -> complex:
    """Utility for the evaluation of the expectation value of a measured state."""
    packed_uint8 = np.packbits(observable.paulis.z, axis=1, bitorder="little")
    state_bytes = np.frombuffer(
        state.to_bytes(packed_uint8.shape[1], "little"), dtype=np.uint8
    )
    reduced = np.bitwise_xor.reduce(packed_uint8 & state_bytes, axis=1)
    return np.sum(observable.coeffs * _PARITY[reduced])


def best_solution(samples, hamiltonian):
    """Find solution with lowest cost"""
    min_cost = 1000
    min_sol = None
    for bit_str in samples.keys():
        # Qiskit use little endian hence the [::-1]
        candidate_sol = int(bit_str)
        # fval = qp.objective.evaluate(candidate_sol)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        if fval <= min_cost:
            min_sol = candidate_sol

    return min_sol


best_sol_100 = best_solution(final_distribution_100_int, cost_hamiltonian_100)
best_sol_bitstring_100 = to_bitstring(int(best_sol_100), len(graph_100))
best_sol_bitstring_100.reverse()

print("Result bitstring:", best_sol_bitstring_100)

Result bitstring: [1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1]


Next, visualize the cut. Nodes of the same color belong to the same group.

In [26]:
plot_result(graph_100, best_sol_bitstring_100)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/b4a25e28-0.avif" alt="Output of the previous code cell" />

#### Hamiltonián &rarr; kvantový Circuit

In [27]:
cut_value_100 = evaluate_sample(best_sol_bitstring_100, graph_100)
print("The value of the cut is:", cut_value_100)

The value of the cut is: 124


![Výstup předchozí buňky kódu](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/9693adfc-0.avif)

### Krok 2: Optimalizace problému pro kvantové spuštění
Pro škálování kroku optimalizace obvodu na problémy utility-scale můžeš využít vysoce výkonné strategie transpilace zavedené v Qiskit SDK v1.0. Mezi další nástroje patří nová služba Transpileru s [AI vylepšenými transpilačními průchody](/guides/ai-transpiler-passes).

In [28]:
# auxiliary function to help plot cumulative distribution functions
def _plot_cdf(objective_values: dict, ax, color):
    x_vals = sorted(objective_values.keys(), reverse=True)
    y_vals = np.cumsum([objective_values[x] for x in x_vals])
    ax.plot(x_vals, y_vals, color=color)


def plot_cdf(dist, ax, title):
    _plot_cdf(
        dist,
        ax,
        "C1",
    )
    ax.vlines(min(list(dist.keys())), 0, 1, "C1", linestyle="--")

    ax.set_title(title)
    ax.set_xlabel("Objective function value")
    ax.set_ylabel("Cumulative distribution function")
    ax.grid(alpha=0.3)


# auxiliary function to convert bit-strings to objective values
def samples_to_objective_values(samples, hamiltonian):
    """Convert the samples to values of the objective function."""

    objective_values = defaultdict(float)
    for bit_str, prob in samples.items():
        candidate_sol = int(bit_str)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        objective_values[fval] += prob

    return objective_values

In [29]:
result_dist = samples_to_objective_values(
    final_distribution_100_int, cost_hamiltonian_100
)

Finally, you can plot the cumulative distribution function to visualize how each sample contributes to the total probability distribution and the corresponding objective value. The horizontal spread shows the range of objective values of the samples in the final distribution. Ideally, you would see that the cumulative distribution function has "jumps" at the lower end of the objective function value axis. This would mean that few solutions with low cost have high probability of being sampled. A smooth, wide curve indicates that each sample is similarly likely, and they can have very different objective values, low or high.

In [30]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plot_cdf(result_dist, ax, "Eagle device")

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/4381a2b3-0.avif" alt="Output of the previous code cell" />

Jakmile jsou nalezeny optimální parametry ze spuštění QAOA na zařízení, přiřaď tyto parametry do Circuit.